# 🔍 Kaggle / FICS PCB Dataset Cleaning & SAM Polygon Point Extractor

This notebook cleans bounding box annotations from the **Kaggle / FICS PCB dataset**, prompts Meta's **Segment Anything Model (SAM ViT-B)** to extract pixel-exact component masks, extracts geometric polygon boundary points `[(x, y), ...]`, and exports:
- 📊 **Excel Spreadsheet (`.xlsx`)** & **CSV (`.csv`)** with all coordinates and KiCad footprints
- 🏷️ **LabelMe JSON (`.json`)** with `ref_des: class` tags ready for visual inspection
- 🏷️ **YOLO-seg (`.txt`)** polygon segmentation labels

In [ ]:
# 1. Install & Import Dependencies
!pip install -q opencv-python numpy pandas openpyxl matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import json
import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

In [ ]:
# 2. Download Pretrained SAM ViT-B Weights (if not present)
import urllib.request

SAM_CHECKPOINT = Path('sam_vit_b.pth')
SAM_URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

if not SAM_CHECKPOINT.exists():
    print('Downloading SAM ViT-B weights (375 MB)...')
    urllib.request.urlretrieve(SAM_URL, str(SAM_CHECKPOINT))
    print('Download complete!')
else:
    print('SAM ViT-B weights already present locally.')

In [ ]:
# 3. Configuration & KiCad Library Convention (KLC) Class Map
CLASS_MAP = {
    0: 'Cap1', 1: 'Cap2', 2: 'Cap3', 3: 'Cap4',
    4: 'MOSFET', 5: 'Mov', 6: 'Resistor', 7: 'Transformer'
}

PREFIX_MAP = {
    'Cap1': 'C', 'Cap2': 'C', 'Cap3': 'C', 'Cap4': 'C',
    'MOSFET': 'Q', 'Mov': 'D', 'Resistor': 'R', 'Transformer': 'T'
}

KICAD_FOOTPRINTS = {
    'Resistor': 'Resistor_SMD:R_0805_2012Metric',
    'Cap1': 'Capacitor_SMD:C_0805_2012Metric',
    'Cap2': 'Capacitor_SMD:C_1206_3216Metric',
    'Cap3': 'Capacitor_THT:CP_Radial_D6.3mm_P2.50mm',
    'Cap4': 'Capacitor_THT:CP_Radial_D8.0mm_P3.50mm',
    'MOSFET': 'Package_TO_SOT_SMD:SOT-23',
    'Mov': 'Diode_SMD:D_SOD-123',
    'Transformer': 'Transformer_SMD:Transformer_Bourns_SRF0703'
}

In [ ]:
# 4. Box Cleaning & Polygon Extraction Algorithms
def clean_box(x1, y1, x2, y2, img_w, img_h, min_size=8):
    if x1 > x2: x1, x2 = x2, x1
    if y1 > y2: y1, y2 = y2, y1
    x1 = max(0, min(int(round(x1)), img_w - 1))
    y1 = max(0, min(int(round(y1)), img_h - 1))
    x2 = max(0, min(int(round(x2)), img_w))
    y2 = max(0, min(int(round(y2)), img_h))
    if (x2 - x1) < min_size or (y2 - y1) < min_size:
        return False, []
    return True, [x1, y1, x2, y2]

def extract_polygon_points(mask: np.ndarray, min_area: float = 15.0):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area:
        return []
    epsilon = 0.005 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    return [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]

In [ ]:
# 5. Run SAM on PCB Image, Extract Points, and Save to Excel & CSV
# Set your image and label paths
image_path = Path('dataset_split/train/images/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.jpg')
labels_path = Path('dataset_split/train/labels/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.txt')
output_dir = Path('kaggle_sam_output')
output_dir.mkdir(parents=True, exist_ok=True)

img = cv2.imread(str(image_path))
h, w = img.shape[:2]

# Load SAM
sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CHECKPOINT))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)
predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

# Parse YOLO bboxes
boxes = []
with open(labels_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            cid = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = (xc - bw / 2.0) * w
            y1 = (yc - bh / 2.0) * h
            x2 = (xc + bw / 2.0) * w
            y2 = (yc + bh / 2.0) * h
            ok, box = clean_box(x1, y1, x2, y2, w, h)
            if ok:
                boxes.append((box, cid))

records = []
shapes = []
ref_counts = {}
vis_img = img.copy()

for idx, (b, cid) in enumerate(boxes, 1):
    input_box = np.array(b)
    masks, scores, _ = predictor.predict(box=input_box[None, :], multimask_output=False)
    mask = masks[0]
    conf = float(scores[0])
    pts = extract_polygon_points(mask)
    if not pts:
        pts = [[float(b[0]), float(b[1])], [float(b[2]), float(b[1])], [float(b[2]), float(b[3])], [float(b[0]), float(b[3])]]
    
    class_name = CLASS_MAP.get(cid, f'Class_{cid}')
    prefix = PREFIX_MAP.get(class_name, 'U')
    ref_counts[prefix] = ref_counts.get(prefix, 0) + 1
    ref_des = f'{prefix}{ref_counts[prefix]}'
    
    records.append({
        'image_name': image_path.name,
        'instance_id': idx,
        'ref_des': ref_des,
        'class_name': class_name,
        'footprint': KICAD_FOOTPRINTS.get(class_name, 'Unknown'),
        'confidence': round(conf, 3),
        'num_points': len(pts),
        'polygon_points_compact': '; '.join([f'({p[0]},{p[1]})' for p in pts]),
        'polygon_points_json': json.dumps(pts)
    })
    
    shapes.append({
        'label': f'{ref_des}: {class_name}',
        'points': pts,
        'shape_type': 'polygon'
    })
    
    cnt = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(vis_img, [cnt], True, (0, 255, 255), 2)

df = pd.DataFrame(records)
df.to_excel(output_dir / 'kaggle_sam_points.xlsx', index=False)
df.to_csv(output_dir / 'kaggle_sam_points.csv', index=False)
print('Saved Excel & CSV!')
df.head()

In [ ]:
# 6. Visualize SAM Overlaid Polygons Inline
plt.figure(figsize=(12, 10))
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.title('SAM Polygon Masks & Contours on PCB')
plt.axis('off')
plt.show()